# Flatten executions
Project nested book deltas into the canonical `Execution` table with Arrow kernels.

In [ ]:
source = "market.books"
target = "market.executions"
start = None
end = None
catalog = {"name": "rekep", "properties": {}}
table_properties = {"history.expire.max-snapshot-age-ms": "604800000"}
branch = "root"
merge_by = True
commit_batch_num = 8
commit_row_size = None
log_level = "INFO"

In [ ]:
from pyiceberg.expressions import And, GreaterThanOrEqual, LessThan

from rekep.iceberg import IcebergCatalog
from rekep.logs import Stage, configure
from rekep.market import Book, Execution
from rekep.times import unix_of

configure(log_level)
if isinstance(commit_batch_num, bool) or not isinstance(commit_batch_num, int):
    raise TypeError("commit_batch_num must be an integer")
if commit_batch_num <= 0:
    raise ValueError("commit_batch_num must be positive")
if commit_row_size is not None and (
    isinstance(commit_row_size, bool) or not isinstance(commit_row_size, int)
):
    raise TypeError("commit_row_size must be an integer or null")
if commit_row_size is not None and commit_row_size <= 0:
    raise ValueError("commit_row_size must be positive")

def _filter(column="unix"):
    lower, upper = unix_of(start), unix_of(end, upper=True)
    predicates = []
    if lower is not None:
        predicates.append(GreaterThanOrEqual(column, lower))
    if upper is not None:
        predicates.append(LessThan(column, upper))
    return (
        None
        if not predicates
        else predicates[0]
        if len(predicates) == 1
        else And(*predicates)
    )


store = IcebergCatalog.from_dict(catalog)
books = store.dataset(
    source,
    field=Book.into_field(),
    branch=branch,
)
executions = store.dataset(
    target,
    field=Execution.into_field(),
    table_properties=dict(table_properties),
    branch=branch,
    commit_batch_num=commit_batch_num,
    commit_row_size=commit_row_size,
)
stage = Stage(
    "flatten_executions",
    sources={"books": source},
    targets={"executions": target},
    window=(unix_of(start), unix_of(end, upper=True)),
)
counts = {"read": 0}


def _batches():
    reader = books.read_arrow_reader(
        Book.into_field(), row_filter=_filter(), order_by=("unix", "hash")
    )
    for batch in reader:
        flattened = Execution.from_books_arrow_batch(batch)
        counts["read"] += flattened.num_rows
        if flattened.num_rows:
            yield flattened


written = executions.append_arrow_reader(
    _batches(),
    Execution.into_field(),
    merge_by=merge_by,
    commit_row_size=commit_row_size,
    commit_batch_num=commit_batch_num,
)
stage.says("projected %d executions out of the books in the window", counts["read"])
result = stage.finished(read=counts["read"], written=written)
result